# Chapter 8: Reinforcement Learning

Paired notebook for Chapter 8 of *Inductive Biases in Neural Networks*, the
book's closing chapter. It runs the one experiment the chapter rests on: a
REINFORCE policy-gradient agent learning to cross a $5 \times 5$ gridworld,
implemented in pure NumPy in `examples/reinforce_gridworld.py`.

**This notebook is NumPy-only. It does not import `torch`.** The gridworld
run is cheap (roughly a minute, fully seeded), so every number the chapter
prose quotes is regenerated here live, not cited from a recorded table.

**Re-run here (NumPy):**
1. The untrained policy's before-training behaviour (mean return, episode
   length, success rate).
2. A 2000-episode REINFORCE run ($\gamma = 0.99$, learning rate $0.05$, a
   moving-average return baseline), recording the smoothed learning curve.
3. The trained policy's after-training behaviour and its greedy trajectory.

The policy network, `PolicyMLP`, is a one-hidden-layer MLP with a $\tanh$
hidden layer and a softmax over four actions: deliberately the same
`Linear` $\to$ `Tanh` $\to$ `Linear` $\to$ `SoftmaxCrossEntropy` stack the
`scratchnn` chapters built, just vectorized in NumPy so a few times $10^4$
forward passes run in seconds. The point of the chapter is that its update
rule, the softmax cross-entropy gradient (predicted distribution minus the
one-hot of the chosen action) scaled by the return, is the supervised
gradient already derived in Chapter 1.

**Seeds:** the policy is built with `seed=0`; training uses `seed=0` for the
rollout sampler; the before/after evaluations use `seed=999`; the sampled
pre-training trajectory uses `seed=42`. All fixed, so the run is
reproducible.

## Setup

Everything comes from `examples/reinforce_gridworld.py`: the environment
(`step`, `one_hot`, `state_index`), the policy (`PolicyMLP`), the REINFORCE
loop (`rollout`, `returns_to_go`, `train`), and the evaluators (`evaluate`,
`greedy_trajectory`, `sample_trajectory`). `import numpy` is the only
heavyweight import; there is deliberately no `import torch` anywhere.

In [1]:
import os
import sys

import numpy as np

# The NumPy gridworld and the REINFORCE loop live in the examples/ directory
# of the parent scratchnn repo.
EXAMPLES = "/home/spinoza/github/repos/scratchnn/examples"
sys.path.insert(0, EXAMPLES)

import reinforce_gridworld as gw
from reinforce_gridworld import (
    PolicyMLP, rollout, returns_to_go, train,
    evaluate, greedy_trajectory, sample_trajectory,
    GRID, N_STATES, N_ACTIONS, START, GOAL,
    STEP_REWARD, GOAL_REWARD, MAX_STEPS, GAMMA,
)

# Guard the no-torch invariant: this notebook must never import torch.
assert "torch" not in sys.modules, "this notebook is NumPy-only; torch must not be imported"

print("numpy", np.__version__)
print("no torch imported:", "torch" not in sys.modules)
print(f"grid {GRID}x{GRID}  start {START}  goal {GOAL}")
print(f"reward {STEP_REWARD}/step + {GOAL_REWARD} at goal  "
      f"gamma {GAMMA}  max steps {MAX_STEPS}")

numpy 2.2.6
no torch imported: True
grid 5x5  start (0, 0)  goal (4, 4)
reward -0.01/step + 1.0 at goal  gamma 0.99  max steps 25


## The environment and the optimal return

The agent starts at $(0, 0)$ and must reach the goal $(4, 4)$. Each step
costs $-0.01$; entering the goal pays $+1.0$ and ends the episode, which
otherwise ends after 25 steps. The shortest path is the Manhattan distance,
$4 + 4 = 8$ steps, so the optimal episode pays seven step-penalties before
the goal-reward step:

$$R^\star = 1.0 - 0.01 \times 7 = 0.93.$$

That $+0.93$ is the ceiling the learning curve should approach.

In [2]:
manhattan = (GOAL[0] - START[0]) + (GOAL[1] - START[1])
optimal_return = GOAL_REWARD + STEP_REWARD * (manhattan - 1)
print(f"Manhattan distance start->goal : {manhattan} steps")
print(f"optimal return  1.0 - 0.01*{manhattan-1} = {optimal_return:+.2f}")

Manhattan distance start->goal : 8 steps
optimal return  1.0 - 0.01*7 = +0.93


## Before training: the untrained policy wanders

With the policy network freshly initialized (`seed=0`), the action
distribution at every state is close to uniform over the four moves. We
evaluate it over 200 sampled trajectories (`seed=999`). The expected return
is negative: most episodes pay the per-step cost and hit the 25-step limit
without ever stumbling into the goal.

In [3]:
init_policy = PolicyMLP(seed=0)
pre = evaluate(init_policy)
print("Before training (untrained policy):")
print(f"  mean return  {pre['mean_return']:+.4f}")
print(f"  mean length  {pre['mean_length']:5.2f} steps")
print(f"  success rate {pre['success_rate']:.1%}")

Before training (untrained policy):
  mean return  -0.1258
  mean length  24.20 steps
  success rate 11.5%


In [4]:
# A single sampled trajectory under the untrained policy (seed=42): it
# bounces around the grid and runs out the 25-step budget.
rng_demo = np.random.default_rng(42)
cells_pre, total_pre = sample_trajectory(init_policy, rng_demo)
print(f"sample pre-training trajectory: {len(cells_pre)-1} steps, "
      f"return {total_pre:+.4f}")
print(f"  reaches goal: {cells_pre[-1] == GOAL}")
print(f"  path: {cells_pre}")

sample pre-training trajectory: 25 steps, return -0.2500
  reaches goal: False
  path: [(0, 0), (0, 0), (1, 0), (1, 1), (1, 0), (0, 0), (0, 1), (0, 0), (0, 1), (0, 1), (1, 1), (2, 1), (2, 2), (2, 1), (2, 2), (3, 2), (2, 2), (2, 1), (1, 1), (1, 2), (1, 1), (1, 0), (0, 0), (0, 1), (0, 2), (0, 3)]


## Training: 2000 episodes of REINFORCE

The training loop (`train`, reproduced from the source file) is one screen
of code. For each episode it rolls out under the current policy, computes the
returns-to-go $G_t$ by a single backward pass over the rewards, subtracts a
moving-average baseline to get the per-step weights $w_t = G_t - b$, and
takes one SGD step on $-\sum_t \log \pi_\theta(a_t \mid s_t)\, w_t$. The
per-step gradient of that objective is exactly the softmax cross-entropy
gradient, the predicted distribution minus the one-hot chosen action, scaled
by $w_t$: this is the `grads_for_episode` method, whose body is

```
dlogits = P.copy()                 # P = softmax(logits), shape (T, 4)
dlogits[arange(T), actions] -= 1.0 # p - onehot(a) at each step
dlogits *= weights[:, None]        # scale each step by w_t = G_t - baseline
```

which is the chapter's bridge in code. Training prints the mean return over
the last 100 episodes every 100 episodes; we keep that smoothed curve for the
figure.

In [5]:
policy, history, smoothed = train(n_episodes=2000, lr=0.05, seed=0,
                                  log_every=100)
print()
print(f"trained on {len(history)} episodes; "
      f"{len(smoothed)} smoothed checkpoints recorded")

episode   100  mean return (last 100): +0.7357
episode   200  mean return (last 100): +0.9020
episode   300  mean return (last 100): +0.9104
episode   400  mean return (last 100): +0.9162


episode   500  mean return (last 100): +0.9177


episode   600  mean return (last 100): +0.9192
episode   700  mean return (last 100): +0.9192
episode   800  mean return (last 100): +0.9218
episode   900  mean return (last 100): +0.9227
episode  1000  mean return (last 100): +0.9222


episode  1100  mean return (last 100): +0.9228


episode  1200  mean return (last 100): +0.9253
episode  1300  mean return (last 100): +0.9250
episode  1400  mean return (last 100): +0.9235
episode  1500  mean return (last 100): +0.9267
episode  1600  mean return (last 100): +0.9266


episode  1700  mean return (last 100): +0.9261


episode  1800  mean return (last 100): +0.9268
episode  1900  mean return (last 100): +0.9286
episode  2000  mean return (last 100): +0.9279

trained on 2000 episodes; 20 smoothed checkpoints recorded


## After training: a near-optimal policy

The same 200-trajectory evaluation (`seed=999`), now under the trained
policy. Every sampled trajectory reaches the goal, and the mean length is
barely above the optimal 8 steps.

In [6]:
post = evaluate(policy)
print("After 2000 episodes of REINFORCE:")
print(f"  mean return  {post['mean_return']:+.4f}")
print(f"  mean length  {post['mean_length']:5.2f} steps")
print(f"  success rate {post['success_rate']:.1%}")

After 2000 episodes of REINFORCE:
  mean return  +0.9278
  mean length   8.21 steps
  success rate 100.0%


In [7]:
# The greedy (argmax-action) trajectory: the policy walks a shortest path.
cells_post, total_post = greedy_trajectory(policy)
arrow_path = " -> ".join(str(c) for c in cells_post)
print(f"greedy trajectory: {len(cells_post)-1} steps, return {total_post:+.4f}")
print(f"  reaches goal: {cells_post[-1] == GOAL}")
print(f"  optimal length: {cells_post[-1] == GOAL and (len(cells_post)-1) == manhattan}")
print(f"  path: {arrow_path}")

greedy trajectory: 8 steps, return +0.9300
  reaches goal: True
  optimal length: True
  path: (0, 0) -> (1, 0) -> (2, 0) -> (2, 1) -> (2, 2) -> (2, 3) -> (3, 3) -> (3, 4) -> (4, 4)


## The smoothed learning curve

Mean return per 100 episodes across the 2000-episode run. The shape is the
one every policy-gradient run on a sparse-reward task has: a fast climb in
the first roughly 200 episodes (the first chance trajectory that reaches the
goal puts strong positive weight on the actions that got there), then a slow
refinement as the policy concentrates on the shortest paths.

In [8]:
print("episode   mean return (smoothed over 100)")
for ep, ret in smoothed:
    print(f"  {ep:5d}   {ret:+.4f}")

episode   mean return (smoothed over 100)
    100   +0.7357
    200   +0.9020
    300   +0.9104
    400   +0.9162
    500   +0.9177
    600   +0.9192
    700   +0.9192
    800   +0.9218
    900   +0.9227
   1000   +0.9222
   1100   +0.9228
   1200   +0.9253
   1300   +0.9250
   1400   +0.9235
   1500   +0.9267
   1600   +0.9266
   1700   +0.9261
   1800   +0.9268
   1900   +0.9286
   2000   +0.9279


### Figure: the learning curve

Mean return against episode, with the optimal return ($+0.93$) and the
untrained baseline drawn for reference. Saved to
`../book/figures/ch08-gridworld-learning-curve.pdf` and placed in the
chapter's worked-example section.

In [9]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIG = "../book/figures/ch08-gridworld-learning-curve.pdf"
eps = [e for e, _ in smoothed]
rets = [r for _, r in smoothed]

fig, ax = plt.subplots(figsize=(6.0, 4.0))
ax.plot(eps, rets, marker="o", color="#4C72B0", label="mean return (per 100 ep)")
ax.axhline(optimal_return, linestyle="--", color="#55A868",
           label=f"optimal return (+{optimal_return:.2f})")
ax.axhline(pre["mean_return"], linestyle=":", color="#C44E52",
           label=f"untrained ({pre['mean_return']:+.3f})")
ax.set_xlabel("episode")
ax.set_ylabel("mean return")
ax.set_title(r"REINFORCE on the $5\times5$ gridworld (2000 episodes)")
ax.set_ylim(min(pre["mean_return"], min(rets)) - 0.1, optimal_return + 0.1)
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()
os.makedirs(os.path.dirname(FIG), exist_ok=True)
fig.savefig(FIG)
print(f"saved {FIG}")

saved ../book/figures/ch08-gridworld-learning-curve.pdf


## Results

Everything the chapter prose quotes, in one place. Every number below is a
live NumPy re-run of `examples/reinforce_gridworld.py` (seeds as documented
at the top); there is no cited or recorded fallback for this chapter.

In [10]:
print("=" * 60)
print("RE-RUN (NumPy, this notebook) -- REINFORCE gridworld")
print("=" * 60)
print(f"grid {GRID}x{GRID}, start {START}, goal {GOAL}, "
      f"reward {STEP_REWARD}/step + {GOAL_REWARD} at goal")
print(f"gamma {GAMMA}, max steps {MAX_STEPS}, lr 0.05, 2000 episodes, seed 0")
print()
print(f"optimal return (8-step Manhattan path) : {optimal_return:+.2f}")
print()
print("Before training (untrained policy):")
print(f"  mean return  {pre['mean_return']:+.4f}")
print(f"  mean length  {pre['mean_length']:5.2f} steps")
print(f"  success rate {pre['success_rate']:.1%}")
print()
print("After 2000 episodes:")
print(f"  mean return  {post['mean_return']:+.4f}")
print(f"  mean length  {post['mean_length']:5.2f} steps")
print(f"  success rate {post['success_rate']:.1%}")
print()
print("Greedy trajectory (argmax actions):")
print(f"  {len(cells_post)-1} steps, return {total_post:+.4f}")
print(f"  {' -> '.join(str(c) for c in cells_post)}")
print()
print("Smoothed learning curve (mean return per 100 episodes):")
for ep, ret in smoothed:
    print(f"  episode {ep:5d}: {ret:+.4f}")
print()
print("Figure written:")
print(f"  {FIG}")

RE-RUN (NumPy, this notebook) -- REINFORCE gridworld
grid 5x5, start (0, 0), goal (4, 4), reward -0.01/step + 1.0 at goal
gamma 0.99, max steps 25, lr 0.05, 2000 episodes, seed 0

optimal return (8-step Manhattan path) : +0.93

Before training (untrained policy):
  mean return  -0.1258
  mean length  24.20 steps
  success rate 11.5%

After 2000 episodes:
  mean return  +0.9278
  mean length   8.21 steps
  success rate 100.0%

Greedy trajectory (argmax actions):
  8 steps, return +0.9300
  (0, 0) -> (1, 0) -> (2, 0) -> (2, 1) -> (2, 2) -> (2, 3) -> (3, 3) -> (3, 4) -> (4, 4)

Smoothed learning curve (mean return per 100 episodes):
  episode   100: +0.7357
  episode   200: +0.9020
  episode   300: +0.9104
  episode   400: +0.9162
  episode   500: +0.9177
  episode   600: +0.9192
  episode   700: +0.9192
  episode   800: +0.9218
  episode   900: +0.9227
  episode  1000: +0.9222
  episode  1100: +0.9228
  episode  1200: +0.9253
  episode  1300: +0.9250
  episode  1400: +0.9235
  episode  1